# Exploration des résultats du pipeline Zenith

Ce notebook lit les sorties produites par `scripts/run_pipeline.py` et illustre :
1. La répartition ABC × XYZ
2. La détection d'obsolescence
3. Les performances de prévision par classe
4. La comparaison entre la politique empirique et la politique optimisée

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

clean = pd.read_parquet(ROOT / 'data/processed/transactions_clean.parquet')
products = pd.read_csv(ROOT / 'data/results/obsolescence.csv')
metrics = pd.read_csv(ROOT / 'data/results/forecast_metrics.csv')
metrics_by_class = pd.read_csv(ROOT / 'data/results/forecast_metrics_by_class.csv')
plan = pd.read_csv(ROOT / 'data/results/optimization_plan.csv')
compare = pd.read_csv(ROOT / 'data/results/financial_comparison.csv')

print(f'Transactions nettoyées : {len(clean):,}'.replace(',', ' '))
print(f'Produits : {len(products)}')
print(f'Période : {clean.date.min().date()} → {clean.date.max().date()}')

## 1. Classification ABC × XYZ

In [ ]:
products.groupby(['classe_abc','classe_xyz']).size().unstack(fill_value=0)

## 2. Détection d'obsolescence

In [ ]:
products['a_risque_obsolescence'].value_counts(normalize=True).rename({0:'actif', 1:'à risque'})

In [ ]:
products[products.a_risque_obsolescence==1][['produit_id','famille','classe_abc','jours_depuis_derniere_vente','ratio_ventes_3m_vs_12m']].head(15)

## 3. Précision des modèles de prévision (par classe ABC)

In [ ]:
metrics_by_class

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
sns.boxplot(data=metrics.dropna(subset=['mae']), x='classe_abc', y='mae', hue='modele', order=['A','B','C'], ax=ax)
ax.set_title('MAE par classe ABC et modèle')
plt.show()

## 4. Politique empirique vs politique optimisée

In [ ]:
compare

In [ ]:
agg = plan.groupby('classe_abc').agg(
    demande=('demande_prevue','sum'),
    qte=('quantite_commandee','sum'),
    rupture=('rupture','sum'),
)
agg['taux_service'] = (1 - agg['rupture'] / agg['demande']) * 100
agg.round(1)